# 02 — Gemma 3 Architecture Lab

Prototype new layers and attention variants using the building blocks in
`src/core/`. This is where you iterate before promoting to `src/`.

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
from src.models.gemma3.config import Gemma3Config
from src.models.gemma3.model import Gemma3SLM
from src.core.generation import generate
from src.utils.training import count_params

In [ ]:
# ── Build a tiny model for fast iteration ────────────────────────────────
tiny_cfg = Gemma3Config(
    vocab_size=50257, block_size=32,
    n_layer=2, n_head=2, n_embd=64,
    dropout=0.0
)
model = Gemma3SLM(tiny_cfg)
print(f'Parameters: {count_params(model):,}')

In [ ]:
# ── Verify forward pass shapes ────────────────────────────────────────────
B, T = 2, tiny_cfg.block_size
idx = torch.randint(0, tiny_cfg.vocab_size, (B, T))
targets = torch.randint(0, tiny_cfg.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f'logits: {logits.shape}  |  loss: {loss.item():.4f}')

In [ ]:
# ── Test generation ───────────────────────────────────────────────────────
prompt = torch.randint(0, tiny_cfg.vocab_size, (1, 4))
output = generate(model, prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f'Input tokens:  {prompt[0].tolist()}')
print(f'Output tokens: {output[0].tolist()}')

In [ ]:
# ── Parameter count comparison across model sizes ────────────────────────
configs = {
    'gemma3_tiny':   Gemma3Config(vocab_size=50257, block_size=16,  n_layer=2,  n_head=2,  n_embd=64),
    'gemma3_small':  Gemma3Config(vocab_size=50257, block_size=128, n_layer=6,  n_head=6,  n_embd=384),
    'gemma3_medium': Gemma3Config(vocab_size=50257, block_size=256, n_layer=12, n_head=12, n_embd=768),
}
for name, cfg in configs.items():
    n = count_params(Gemma3SLM(cfg))
    print(f'{name:14s}: {n:>12,} params  ({n/1e6:.1f}M)')